# 03 - Forecasting, Evaluated Honestly

**Business question:** the app currently forecasts prices 1-5 years ahead with a straight line. Is that useful? Can any model beat "tomorrow = today"? And what *can* be forecast?

Method: **walk-forward validation**. At each origin the model sees only past data, forecasts `h` days ahead, and is scored against what really happened. Every model is compared with a naive baseline - a model that can't beat naive has no skill, however good its chart looks.

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "analytics").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from analytics import data, metrics, stats_tests, forecasting
from analytics.portfolio import xirr, fifo_realized_pnl

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 4.5)
pd.options.display.float_format = "{:,.4f}".format

In [ ]:
ohlcv = data.load_ohlcv()
prices = ohlcv.pivot(index="date", columns="symbol", values="close").sort_index()
volume = ohlcv.pivot(index="date", columns="symbol", values="volume").sort_index()
sources = data.data_sources()
print("Data source(s):", sources, "|", prices.shape[1], "coins |", prices.index.min().date(), "->", prices.index.max().date())
if "synthetic" in sources:
    print("\n*** WARNING: SYNTHETIC DEMO DATA. Re-run `python -m analytics.etl.ingest` for real data before reading anything into the results. ***")

In [ ]:
SYMBOLS = [s for s in ["BTC", "ETH"] if s in prices.columns] or list(prices.columns[:2])
HORIZONS = [7, 30]
STEP = 14          # days between forecast origins; lower = more forecasts but slower

all_forecasts, all_results = {}, {}
for s in SYMBOLS:
    for h in HORIZONS:
        fcs = forecasting.walk_forward(prices[s].dropna(), horizon=h, step=STEP)
        all_forecasts[(s, h)] = fcs
        all_results[(s, h)] = forecasting.evaluate(fcs)
        print(f"done {s} h={h}: {fcs['origin_date'].nunique()} forecast origins")

## Results: error metrics by model

In [ ]:
for (s, h), res in all_results.items():
    print(f"\n=== {s}, {h}-day horizon ===")
    display(res.style.format({"MAE": "{:,.2f}", "RMSE": "{:,.2f}", "MAPE": "{:.1%}", "bias": "{:,.2f}", "directional_accuracy": "{:.1%}", "skill_vs_naive": "{:+.1%}"}))

**Reading the table:** `skill_vs_naive` > 0 means lower MAE than naive. `directional_accuracy` of ~50% means no better than a coin flip at calling up/down.

In [ ]:
fig, axes = plt.subplots(len(SYMBOLS), len(HORIZONS), figsize=(13, 3.6 * len(SYMBOLS)), squeeze=False)
for i, s in enumerate(SYMBOLS):
    for j, h in enumerate(HORIZONS):
        res = all_results[(s, h)].sort_values("skill_vs_naive")
        axes[i][j].barh(res.index, res["skill_vs_naive"], color=["C2" if v > 0 else "C3" for v in res["skill_vs_naive"]])
        axes[i][j].axvline(0, c="k", lw=1); axes[i][j].set_title(f"{s}, {h}d: skill vs naive")
plt.tight_layout(); plt.show()

## What does the original linear-trend method actually do?

In [ ]:
focus = SYMBOLS[0]
fc7 = all_forecasts[(focus, 30)]
pivot = fc7.pivot(index="target_date", columns="model", values="forecast")
actual = fc7.drop_duplicates("target_date").set_index("target_date")["actual"]
plt.figure(figsize=(11, 4.5))
actual.plot(label="actual price", c="k", lw=2)
pivot["naive"].plot(label="naive", ls="--")
pivot["linear_trend (original app)"].plot(label="linear trend (original app)", c="C3")
plt.title(f"{focus}: 30-day-ahead forecasts vs actual"); plt.legend(); plt.show()

# The app's headline feature: 1/2/3/5-year extrapolation from the full history
history = prices[focus].dropna()
last = history.iloc[-1]
horizons_years = {y: forecasting.linear_trend_forecast(history, 365 * y) for y in [1, 2, 3, 5]}
display(pd.DataFrame({"predicted_price": horizons_years, "multiple_of_current_price": {y: v / last for y, v in horizons_years.items()}}))

> **Interpretation:** *(a linear fit through a trending, volatile series extrapolates the past slope forever - what does the 5-year line imply, and how plausible is it?)*

## Prediction intervals: be honest about uncertainty

In [ ]:
rows = []
for s in SYMBOLS:
    for h in HORIZONS:
        for level in [0.5, 0.8, 0.9]:
            rows.append({"symbol": s, "horizon": h, "nominal": level,
                         "actual_coverage": forecasting.interval_coverage(prices[s].dropna(), h, level, step=STEP)})
coverage = pd.DataFrame(rows)
display(coverage.pivot_table(index=["symbol", "horizon"], columns="nominal", values="actual_coverage").style.format("{:.0%}"))
low, high = forecasting.empirical_interval(prices[focus].dropna(), 30, 0.9)
print(f"{focus}: latest close {prices[focus].dropna().iloc[-1]:,.2f}; 90% range 30 days ahead = {low:,.2f} - {high:,.2f}")

## What *is* forecastable: volatility

In [ ]:
vol_results = {}
for s in SYMBOLS:
    tbl = forecasting.walk_forward_volatility(prices[s].dropna(), horizon=30, step=STEP)
    vol_results[s] = forecasting.evaluate_volatility(tbl)
    print(f"\n=== {s}: 30-day volatility forecasts ===")
    display(vol_results[s].style.format("{:.4f}"))

> **Interpretation:** *(do GARCH/EWMA beat a simple trailing window? even a modest gain matters for risk sizing and VaR)*

## Key takeaways (computed)

In [ ]:
lines = []
for (s, h), res in all_results.items():
    best = res.index[0]
    lt = res.loc["linear_trend (original app)"]
    verdict = f"{best} was best (skill {res.loc[best, 'skill_vs_naive']:+.1%} vs naive)"
    lines.append(f"- **{s}, {h}d:** {verdict}; the original linear trend's MAE was {lt['MAE'] / res.loc['naive', 'MAE']:.1f}x the naive baseline's. Best directional accuracy: {res['directional_accuracy'].max():.0%}.")
beaten = sum((res.drop(index='naive')['skill_vs_naive'] > 0.02).any() for res in all_results.values())
lines.append(f"- In {beaten} of {len(all_results)} coin/horizon combinations at least one model beat naive by more than 2%.")
for s, res in vol_results.items():
    base, best = res.loc["trailing_30d", "RMSE"], res["RMSE"].min()
    lines.append(f"- **{s} volatility:** best model {res['RMSE'].idxmin()} cut RMSE by {1 - best / base:.0%} vs the trailing-30-day baseline.")
mult = horizons_years[5] / last
note = " - a NEGATIVE price, which is impossible, so a straight line is the wrong model class for prices." if mult < 0 else "."
lines.append(f"- **5-year linear extrapolation:** the original method implies {mult:.1f}x today's {focus} price{note}")
from IPython.display import Markdown
Markdown("\n".join(lines))